# RP-010 — Google Colab GPU preflight

Before running this notebook, select **Runtime → Change runtime type → GPU**. This notebook does not load a model, start paid resources, expose a public tunnel, or automatically switch models. It checks whether the selected runtime is eligible for an RP-011 model-load test.

In [1]:
from pathlib import Path
import os
import subprocess

REPOSITORY_URL = 'https://github.com/LiemLiemLiem12/pdf-parser.git'
REPOSITORY_ROOT = Path('/content/ATS-CV')
PROJECT_ROOT = REPOSITORY_ROOT / 'PDFPARSER'

if not (PROJECT_ROOT / 'requirements-ml.txt').is_file():
    if REPOSITORY_URL.startswith('REPLACE_'):
        raise RuntimeError('Set REPOSITORY_URL before cloning the project')
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY_ROOT)], check=True)

os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

Project root: /content/ATS-CV/PDFPARSER


In [2]:
%pip install -q -r requirements-dev.txt -r requirements-ml.txt

In [3]:
# Keep checkpoint downloads in the ephemeral Colab runtime by default.
# Change this only when you have explicitly authorized persistent storage.
HF_CACHE_ROOT = Path('/content/huggingface')
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE_ROOT)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_ROOT / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE_ROOT / 'transformers')
print(f'Hugging Face cache: {HF_CACHE_ROOT}')

Hugging Face cache: /content/huggingface


In [4]:
# Optional: use a Colab Secret named HF_TOKEN only for gated/private models.
# The token is never printed. Qwen2.5 public checkpoints normally do not require it.
USE_HF_TOKEN = False
if USE_HF_TOKEN:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if not token:
        raise RuntimeError('HF_TOKEN was requested but is not configured in Colab Secrets')
    os.environ['HF_TOKEN'] = token

In [5]:
!nvidia-smi

Wed Nov 25 15:41:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import json
from app.llm.runtime_preflight import run_preflight

# Qwen14B is always evaluated first. Change this to 'qwen7b' yourself only
# after recording a blocked Qwen14B report. There is no automatic fallback.
MODEL_ALIAS = 'qwen14b'
REPORT_PATH = PROJECT_ROOT / 'artifacts' / 'rp010_preflight.json'
report = run_preflight(MODEL_ALIAS, output_path=REPORT_PATH)
print(json.dumps(report, indent=2, ensure_ascii=False))
print(f'Report saved to: {REPORT_PATH}')

{
  "model_alias": "qwen7b",
  "decision": "supported_for_load_test",
  "compute_dtype": "bfloat16",
  "resource_gates": {
    "min_vram_gib": 8.0,
    "min_ram_gib": 8.0,
    "min_disk_gib": 20.0
  },
  "reasons": [],
  "snapshot": {
    "python": "3.13.16",
    "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
    "packages": {
      "torch": "2.14.1+cu130",
      "transformers": "5.24.1",
      "accelerate": "1.15.0",
      "bitsandbytes": "0.50.2"
    },
    "system_ram_gib": 12.67,
    "disk_free_gib": 112.23,
    "nvidia_smi": {
      "available": true,
      "output": "Tesla T4, 580.82.07, 15360",
      "error": null
    },
    "gpu": {
      "cuda_available": true,
      "cuda_version": "13.0",
      "device_name": "Tesla T4",
      "compute_capability": [
        7,
        5
      ],
      "vram_gib": 14.56,
      "bf16_supported": true,
      "nf4_smoke_ok": true,
      "nf4_smoke_error": null
    }
  },
  "notice": "Passing preflight permits RP-011 model-load testing; it 

In [7]:
if report['decision'] != 'supported_for_load_test':
    raise RuntimeError(
        'RP-010 blocked: ' + '; '.join(report['reasons']) +
        '. Do not start real-model mode on this runtime.'
    )
print('RP-010 passed: this runtime may proceed to the RP-011 model-load test.')

RP-010 passed: this runtime may proceed to the RP-011 model-load test.


In [8]:
# Optional persistence. Enable only for an authorized Google Drive location.
MOUNT_AUTHORIZED_DRIVE = False
if MOUNT_AUTHORIZED_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    destination = Path('/content/drive/MyDrive/ATS-CV/research-artifacts/rp010_preflight.json')
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPORT_PATH, destination)
    print(f'Report copied to: {destination}')

In [9]:
# Prepare the explicit RP-011 model-load configuration.
os.environ['INFERENCE_MODE'] = 'transformers'
os.environ['MODEL_ALIAS'] = MODEL_ALIAS
os.environ['MODEL_REVISION'] = 'main'
os.environ['QUANTIZATION'] = 'nf4-4bit'
print({
    'INFERENCE_MODE': os.environ['INFERENCE_MODE'],
    'MODEL_ALIAS': os.environ['MODEL_ALIAS'],
    'MODEL_REVISION': os.environ['MODEL_REVISION'],
    'QUANTIZATION': os.environ['QUANTIZATION'],
})

{'INFERENCE_MODE': 'transformers', 'MODEL_ALIAS': 'qwen7b', 'MODEL_REVISION': 'main', 'QUANTIZATION': 'nf4-4bit'}


## RP-011 model-load smoke test

Continue only after RP-010 reports `supported_for_load_test`. This downloads the selected checkpoint, loads it once, rejects CPU/disk offload, and prints reproducibility metadata. For recorded experiments, replace `main` with an immutable commit revision.

In [10]:
from app.core.config import Settings
from app.llm.model_manager import ModelManager

settings = Settings(_env_file=None)
manager = ModelManager(settings)
manager.load()
manager.load()  # Idempotent: the checkpoint is not initialized again.
print(json.dumps(manager.metadata, indent=2, ensure_ascii=False))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

{
  "alias": "qwen7b",
  "model_id": "Qwen/Qwen2.5-7B-Instruct",
  "revision": "main",
  "quantization": "nf4-4bit",
  "compute_dtype": "bfloat16",
  "device_map": {
    "": "cuda:0"
  }
}


In [11]:
# Run this cleanup cell only after inspecting the load result.
manager.close()
print(f'Model manager state: {manager.state.value}')

Model manager state: closed
